# 02 — Metrics

**What do these numbers mean, and what can they not tell you?**

Read this before any experiment. Every wrong result in this project came
from reading a metric as the answer to a question it does not answer — a
lookup counter taken for a transfer counter, a whole-card KV figure quoted
for a slice, a local cache hit read as a KV hop.

| section | |
|---|---|
| 1 | the catalogue — every metric, by the question it answers |
| 2 | the SLAs — each target, why that number, which metric proves it |
| 3 | percentiles — why histograms, and what p50/p90/p99 hide |
| 4 | the traps — each one demonstrated against the live cluster |
| 5 | watch them move — own seeded load, with the real Grafana panels |

Independent: drives its own load, needs nothing from 01. Just `make ssh`.


In [ ]:
%load_ext autoreload
%autoreload 2

import os, sys, time, random
sys.path.insert(0, '..')

from utils import Cluster, Report, filler
from prom import (Prom, Grafana, CATALOGUE, LATENCY_HISTOGRAMS,
                  panel_id, panel_queries, metric_name, GRAFANA_URL)
from slos import SLOS, table_rows

cluster = Cluster()
prom    = Prom()
grafana = Grafana(password=os.environ.get('GRAFANA_PASSWORD', ''))
report  = Report('Dodona 02 — metrics', 'what the numbers mean')

report.objective(
    'What does each metric mean, what does good look like, and what can it '
    'NOT tell you?',
    falsified_by='a metric in the catalogue that no longer exists, or a trap '
                 'that no longer reproduces')


## 1. The catalogue

Grouped by the **question**, not the metric name. A name tells you nothing
about whether its value is good.

The last column is the one that matters: **what this metric cannot tell you.**
Twenty of the twenty-three carry one.


In [ ]:
# metric name first, then the value, then what it means — a reader needs
# to know WHICH series they would go and look at.
for question, entries in CATALOGUE.items():
    report.heading(question)
    rows = []
    for label, expr, means, blind in entries:
        got = prom.query(expr)
        value = f'{got[0][1]:,.4g}' if got else ('—' if not prom.up() else 'no data')
        rows.append([metric_name(expr), value, label, means])
    report.table(['metric', 'now', 'answers', 'means'], rows)
    for label, expr, _, blind in entries:
        if blind:
            report.text(f'{metric_name(expr)} — CANNOT TELL YOU: {blind}')


## 2. The SLAs

A measurement is a number. A measurement against a promise is a result.

These live in `benchmarking/slos.py`, so every notebook checks the same ones
and nothing drifts between documents.


In [ ]:
report.heading('2. The SLAs')
rows = []
for key, s in SLOS.items():
    rows.append([s.label, f'{s.target:g}{s.unit}', s.why])
report.table(['promise', 'target', 'why that number'], rows)
report.text('Concurrency is deliberately absent: it is derived in 03-capacity '
            'from a measured saturation curve. A concurrency target picked '
            'before measuring is a wish.')
report.conclusion(
    'Each has a metric that proves it: TTFT and turn from the latency '
    'histograms, preemptions from vllm:num_preemptions_total, shed rate from '
    'orch_shed_total over orch_requests_total.')


## 3. Percentiles

vLLM publishes **histograms**. The percentile comes from the buckets:

```promql
histogram_quantile(0.99, sum by (le) (rate(vllm:time_to_first_token_seconds_bucket[5m])))
```

Sorting ten samples in Python gives a "p99" that is the maximum of ten and
swings wildly between runs. This notebook did that until recently.

**What each hides from the others:** p50 is the typical experience and says
nothing about the tail. p99 is the tail and says nothing about how common it
is. A p50 of 0.3s with a p99 of 8s is a system that works until it doesn't.


In [ ]:
report.heading('3. Percentiles')
rows, series = [], {}
for name in ('ttft', 'tpot', 'e2e', 'queue'):
    q = prom.quantiles(name, qs=(0.5, 0.9, 0.99))
    if q:
        rows.append([name, f'{q.get(0.5,0):.4f}', f'{q.get(0.9,0):.4f}',
                     f'{q.get(0.99,0):.4f}', f'{q.get(0.99,0)/max(q.get(0.5,1e-9),1e-9):.1f}x'])
        series[name] = [q.get(0.5,0), q.get(0.9,0), q.get(0.99,0)]

if rows:
    report.table(['metric', 'p50 s', 'p90 s', 'p99 s', 'p99/p50'], rows,
                 note='The last column is the tail ratio. Near 1 is even; large is bursty.')
    report.bar('Percentiles by metric', ['p50', 'p90', 'p99'], series,
               ylabel='seconds (log)', log=True,
               note='tpot is per OUTPUT TOKEN, so orders of magnitude below e2e')
else:
    report.text('No histogram data yet — section 5 generates some.')


## 4. The traps

Each of these produced a wrong answer in this project. Demonstrated live, so
you can see the two numbers that look alike sitting next to each other.


In [ ]:
report.heading('4. Trap: a lookup is not a transfer')
asked  = prom.query('sum(vllm:external_prefix_cache_queries_total)')
found  = prom.query('sum(vllm:external_prefix_cache_hits_total)')
landed = prom.query('sum(vllm:prompt_tokens_by_source_total{source="external_kv_transfer"})')
v = lambda r: r[0][1] if r else 0
report.table(['', 'value', 'what it proves'], [
    ['connector asked', f'{v(asked):,.0f}', 'decode looked'],
    ['connector FOUND', f'{v(found):,.0f}', 'the lookup matched — NOT that bytes moved'],
    ['tokens that ARRIVED', f'{v(landed):,.0f}', 'the honest one'],
], note='We once reported found as proof the hop worked while the pull was '
        'returning -1 and every transfer failed.')
report.conclusion(
    'found and arrived agree, so the lookup and the transfer are both working.'
    if v(found) > 0 and abs(v(found) - v(landed)) < max(v(found), 1) * 0.1 else
    f'found={v(found):,.0f} but arrived={v(landed):,.0f}. The gap is lookups that '
    'matched and then failed to transfer — check "Num successful transfers" in '
    '`make kv`.')


In [ ]:
report.heading('4. Trap: decode\'s own cache looks like the connector')
local = prom.query('sum(vllm:prompt_tokens_by_source_total{source="local_cache_hit"})')
comp  = prom.query('sum(vllm:prompt_tokens_by_source_total{source="local_compute"})')
report.bar('Where decode gets its prompt KV',
           ['computed', "decode's OWN cache", 'from the connector'],
           {'prompt tokens': [v(comp), v(local), v(landed)]}, ylabel='tokens',
           note='only the third bar is a KV hop')
report.table(['metric', 'value'], [
    ['vllm:prefix_cache_hits_total (LOCAL)', f"{v(prom.query('sum(vllm:prefix_cache_hits_total)')):,.0f}"],
    ['vllm:external_prefix_cache_hits_total (CONNECTOR)', f'{v(found):,.0f}'],
], note='One word apart. A high local rate beside zero external reads exactly '
        'like a working hop, and is the opposite.')


In [ ]:
report.heading('4. Trap: a request count is not a work count')
ptok = prom.query('sum(vllm:prompt_tokens_total)')
report.table(['', 'tokens', ''], [
    ['prompt_tokens_total', f'{v(ptok):,.0f}', 'tokens IN THE REQUEST'],
    ['  of which computed', f'{v(comp):,.0f}', 'work decode actually did'],
    ['  of which cached', f'{v(local):,.0f}', 'served from its own cache'],
    ['  of which transferred', f'{v(landed):,.0f}', 'arrived as KV'],
], note='prompt_tokens_total does not move when the hop starts working — the '
        'gateway sends decode the whole prompt either way. Only the split moves.')
report.heading('4. Trap: whole card is not a slice')
report.table(['', 'KV', 'sequences @16k'], [
    ['the whole H100', '~64 GiB', '~73.5'],
    ['one 12 GiB HAMi slice', '4.10 GiB', '4.7'],
], note='15x apart. CUDA-graph capture scales with available memory, so the '
        'whole-card figure cannot be divided down.')


## 5. Watch them move

Its own seeded load — 8 short requests and 4 long **unique** ones. Unique
matters: a repeated prompt is served from decode's own cache and nothing
crosses between pods, which is how a KV-hop test accidentally measures a
local cache hit.


In [ ]:
report.heading('5. Watch them move')
random.seed(2)
before = {k: v(prom.query(e)) for k, e in {
    'requests': 'sum(orch_requests_total)',
    'transferred': 'sum(vllm:prompt_tokens_by_source_total{source="external_kv_transfer"})',
    'computed': 'sum(vllm:prompt_tokens_by_source_total{source="local_compute"})',
    'preemptions': 'sum(vllm:num_preemptions_total)',
}.items()}

sent, failed = 0, {}
for i in range(8):
    r = cluster.chat('Reply with the single word: ok', max_tokens=8); sent += 1
    if '_error' in r: failed[r['_error']] = failed.get(r['_error'], 0) + 1
for i in range(4):
    r = cluster.chat(f'[{i}-{random.random()}] ' + filler(4000) + '\n\nReply with: ok',
                     max_tokens=8, timeout=180); sent += 1
    if '_error' in r: failed[r['_error']] = failed.get(r['_error'], 0) + 1

time.sleep(20)   # one scrape interval
after = {k: v(prom.query(e)) for k, e in {
    'requests': 'sum(orch_requests_total)',
    'transferred': 'sum(vllm:prompt_tokens_by_source_total{source="external_kv_transfer"})',
    'computed': 'sum(vllm:prompt_tokens_by_source_total{source="local_compute"})',
    'preemptions': 'sum(vllm:num_preemptions_total)',
}.items()}

report.table(['counter', 'before', 'after', 'moved'],
             [[k, f'{before[k]:,.0f}', f'{after[k]:,.0f}', f'{after[k]-before[k]:+,.0f}']
              for k in before],
             note=f'{sent} requests sent, {sum(failed.values())} refused.')
report.verdict(after['requests'] > before['requests'], 'the gateway counter moved')
report.verdict(after['preemptions'] == before['preemptions'],
               'no preemptions under this load')
moved = after['transferred'] - before['transferred']
report.conclusion(
    f'{moved:,.0f} prompt tokens arrived as KV over the wire during this load, '
    f"against {after['computed']-before['computed']:,.0f} computed locally."
    if moved > 0 else
    'No KV crossed between pods during this load. Either the connector is not '
    'transferring, or the prompts were served from decode\'s own cache — '
    'section 4 distinguishes them.')


### The same numbers, as Grafana draws them

Rendered server-side, not redrawn. Needs `GRAFANA_PASSWORD` and the renderer
deployment; falls back to links.


In [ ]:
from pathlib import Path
shots = Path('screenshots'); shots.mkdir(exist_ok=True)
wanted = [('dodona-vllm', 'vllm', 'GPU KV cache'), ('dodona-vllm', 'vllm', 'TTFT'),
          ('dodona-mooncake', 'mooncake', 'DELIVERED'),
          ('dodona-mooncake', 'mooncake', 'Claimed vs delivered'),
          ('dodona-gateway', 'gateway', 'shed')]
saved = []
for uid, dash, title in wanted:
    pid = panel_id(dash, title)
    if pid is None: continue
    out = grafana.render(uid, pid, shots / f'{dash}-{title[:18].lower().replace(" ","-")}.png')
    if out: saved.append(out)

if saved:
    report.verdict(True, f'rendered {len(saved)} Grafana panels')
    from IPython.display import Image, display
    for s in saved: display(Image(filename=str(s)))
else:
    report.text('No panels rendered — set GRAFANA_PASSWORD (printed by '
                'make observability) and check renderer.yaml is deployed.')
    report.links([(d, f'{GRAFANA_URL}/d/{u}?from=now-30m&to=now')
                  for u, d in dict.fromkeys((u, d) for u, d, _ in wanted)])


## Save


In [ ]:
path  = report.save('02-metrics')
fails = report.failures()
print('wrote', path)
print()
if fails:
    for f in fails: print('  open:', f)
else:
    print('Every metric accounted for. Next: 03-capacity')
